Base de Dados:
Os dados a serem utilizados estão disponíveis no portal de dados abertos

#Bibliotecas e Funções

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)



!pip install pyvis
from pyvis.network import Network
import pandas as pd
import random
from google.colab import files

pd.set_option('display.max_colwidth', None)

def make_graph(grules):
  # Criando a rede com cdn_resources configurado para 'remote' e habilitando física
  net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", notebook=True, directed=True, cdn_resources='in_line')

  # Adicionando nós e arestas com setas
  for index, row in grules.iterrows():
      antecedents = list(row['antecedents'])
      consequents = list(row['consequents'])

      # Adicionando cada antecedente como um nó
      for ant in antecedents:
          net.add_node(ant, ant, title=ant)

      # Adicionando cada consequente como um nó
      for con in consequents:
          net.add_node(con, con, title=con)

      # Adicionando arestas para cada par antecedente-consequente
      for ant in antecedents:
          for con in consequents:
              edge_title = f"Confidence: {row['confidence']:.2f}, Lift: {row['lift']:.2f}"
              net.add_edge(ant, con, value=row['confidence'], title=edge_title)



  # Salvando e fazendo o download do grafo
  net.save_graph("supermarket_associations_directed.html")
  files.download("supermarket_associations_directed.html")


import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

pd.set_option('display.max_colwidth', None)

import pandas as pd
import networkx as nx
from pyvis.network import Network
from google.colab import files

def make_graph2(grules):
    G = nx.DiGraph()

    # Adicionando nós e arestas
    for index, row in grules.iterrows():
        # Criando rótulos para os nós de antecedente e consequente
        antecedent_items = sorted(map(str, row['antecedents']))
        consequent_items = sorted(map(str, row['consequents']))
        antecedent_label = ', '.join(antecedent_items)
        consequent_label = ', '.join(consequent_items)

        # Adicionando os nós de antecedente e consequente se ainda não existirem
        if not G.has_node(antecedent_label):
            G.add_node(antecedent_label, title=antecedent_label)
        if not G.has_node(consequent_label):
            G.add_node(consequent_label, title=consequent_label)

        # Adicionando uma aresta entre o antecedente e o consequente
        G.add_edge(
            antecedent_label,
            consequent_label,
            title=f"Confiança: {row['confidence']:.2f}, Lift: {row['lift']:.2f}",
            confidence=row['confidence'],
            lift=row['lift']
        )

    # Criando a rede com pyvis
    net = Network(
        height="750px",
        width="100%",
        bgcolor="#222222",
        font_color="white",
        notebook=True,
        directed=True,
        cdn_resources='in_line'
    )

    # Convertendo o grafo networkx para pyvis
    net.from_nx(G)

    # Ajustando estilos e interações
    for node in net.nodes:
        node_label = node['label']
        node['title'] = node_label
        node['shape'] = 'box'  # Opcional: para melhor visualização de conjuntos de itens

        # Verificando se o nó representa um único item
        if ',' not in node_label:
            # Verificando se o item começa com 'DS_cluster'
            if node_label.startswith('DS_cluster'):
                # Destaque para os nós que atendem ao critério
                node['color'] = 'red'         # Cor diferenciada
                node['font'] = {'size': 14}   # Fonte maior
                node['borderWidth'] = 2       # Borda mais espessa
                node['size'] = 40             # Nó maior
            else:
                # Estilo padrão para nós individuais
                node['color'] = '#1f78b4'
                node['size'] = 5
                node['font'] = {'size': 2}   # Fonte menor
        else:
            # Estilo para nós que representam múltiplos itens
            node['color'] = '#6a3d9a'
            node['size'] = 5
            node['font'] = {'size': 2}   # Fonte menor

    for edge in net.edges:
        edge['title'] = edge['title']
        edge['value'] = edge['confidence']  # Ajusta a espessura da aresta com base na confiança

    # Habilitando a física para movimentação dinâmica dos nós
    net.show_buttons(filter_=['physics'])
    net.toggle_physics(True)

    # Salvando e fazendo o download do grafo
    net.save_graph("supermarket_associations_directed.html")
    files.download("supermarket_associations_directed.html")




In [ ]:

import zipfile
def descompactar_arquivo(nome_arquivo_zip, destino):
  """
  Descompacta um arquivo zip.

  Args:
    nome_arquivo_zip: O nome do arquivo zip a ser descompactado.
    destino: O diretório onde os arquivos serão extraídos.
  """
  try:
    with zipfile.ZipFile(nome_arquivo_zip, 'r') as zip_ref:
      zip_ref.extractall(destino)
    print(f"Arquivo {nome_arquivo_zip} descompactado com sucesso para {destino}")
  except FileNotFoundError:
    print(f"Arquivo {nome_arquivo_zip} não encontrado.")
  except Exception as e:
    print(f"Erro ao descompactar o arquivo: {e}")


import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib import colormaps

def make_spider(df_plot):

    # Seleciona apenas as colunas numéricas, excluindo a coluna "cluster" e "count"
    num_col = df_plot.select_dtypes(include=['float64', 'int64']).columns.difference(["cluster", "count"])

    n_cols = df_plot['cluster'].unique().shape[0]

    # Normaliza as colunas numéricas
    df_plot.loc[:, num_col] = df_plot[num_col].div(df_plot[num_col].max(axis=0))
    df_plot = df_plot.fillna(0)

    # Seleciona a paleta de cores
    palette = colormaps.get_cmap("Set2")  # Corrigido para aceitar apenas dois argumentos

    # Definir categorias sem a coluna "cluster" e "count"
    categories = list(df_plot[num_col].columns)
    N = len(categories)

    # Ângulos para o gráfico polar
    angles = [n / float(N) * 2 * math.pi for n in range(N)]
    angles += angles[:1]

    # Configurações da figura
    my_dpi = 96
    fig, axes = plt.subplots(df_plot.shape[0] // n_cols, n_cols, figsize=(1500 / my_dpi, 1500 / my_dpi), dpi=my_dpi, subplot_kw={"projection": "polar"})
    axes = axes.ravel()

    for idx, ax in enumerate(axes):
        color = palette(idx)

        # Seleciona a linha atual
        row = df_plot.iloc[idx]

        # Configurações do gráfico polar
        ax.set_theta_offset(math.pi / 2)
        ax.set_theta_direction(-1)

        ax.set_xticks(angles[:-1], categories, color="grey", size=7)

        ax.set_rlabel_position(0)
        ax.set_yticks([0.33, 0.66, 0.99], ["0.3", "0.6", "1"], color="grey", size=10)
        ax.set_ylim(0, 1)

        # Valores para o gráfico, sem as colunas "cluster" e "count"
        values = row[num_col].values.flatten().tolist()
        values += values[:1]

        # Plotar o gráfico
        ax.plot(angles, values, color=color, linewidth=1.5, linestyle='solid')
        ax.fill(angles, values, color=color, alpha=0.4)

        # Título com o número de registros (coluna "count")
        ax.set_title(f"Persona {row['cluster']} - {int(row['count'])} registros", size=14, color=color, y=1.1)

    plt.tight_layout()
    plt.show()

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_boxplots(df, columns, outline=True):
    """
    Plota boxplots verticais para variáveis numéricas selecionadas em um DataFrame,
    exibe os valores dos quartis e a média.

    Parâmetros:
    -----------
    df : pandas.DataFrame
        O DataFrame contendo os dados.
    columns : list
        Lista de nomes de colunas que se deseja plotar.
    outline : bool, opcional (default=True)
        Se True, inclui outliers no boxplot.
    """
    # Verificar se as colunas fornecidas são numéricas
    num_columns = [col for col in columns if pd.api.types.is_numeric_dtype(df[col])]

    # Checa se existem colunas numéricas para plotar
    if not num_columns:
        raise ValueError("Nenhuma das colunas fornecidas é numérica.")

    # Criar os boxplots
    plt.figure(figsize=(4, len(num_columns) * 4))  # Ajuste de tamanho baseado no número de colunas

    for i, col in enumerate(num_columns, 1):
        plt.subplot(len(num_columns), 1, i)

        # Criar o boxplot vertical
        sns.boxplot(data=df, y=col, showfliers=outline, orient="v")

        # Calcular os quartis e a média
        quartile_25 = df[col].quantile(0.25)
        quartile_50 = df[col].median()  # ou quantile(0.50)
        quartile_75 = df[col].quantile(0.75)

        # Calcular os limites para os outliers
        iqr = quartile_75 - quartile_25
        lower_bound = quartile_25 - 1.5 * iqr
        upper_bound = quartile_75 + 1.5 * iqr

        # Exibir os limites dos outliers
        plt.text(0, lower_bound, f'Início Outliers Inf: {lower_bound:,.2f}', horizontalalignment='center', color='purple', weight='bold')
        plt.text(0, upper_bound, f'Início Outliers Sup: {upper_bound:,.2f}', horizontalalignment='center', color='purple', weight='bold')


        # Plotar os valores dos quartis e da média
        plt.text(0, quartile_25, f'Q1: {quartile_25:,.2f}', horizontalalignment='center', color='blue', weight='bold')
        plt.text(0, quartile_50, f'Q2 (Mediana): {quartile_50:,.2f}', horizontalalignment='center', color='blue', weight='bold')
        plt.text(0, quartile_75, f'Q3: {quartile_75:,.2f}', horizontalalignment='center', color='blue', weight='bold')

        # Título e rótulo do eixo Y
        plt.title(f'Boxplot de {col}')
        plt.ylabel(col)

    plt.tight_layout()
    plt.show()

import matplotlib.pyplot as plt
# def plot_pie_charts(df, category_col, split_col=None, color_dict=None ):
#     """
#     Plota gráficos de pizza. Se `split_col` for fornecido, cria um gráfico de pizza
#     para cada valor único em `split_col`, caso contrário, cria um único gráfico de pizza.

#     Parâmetros:
#     -----------
#     df : pandas.DataFrame
#         O DataFrame contendo os dados.
#     category_col : str
#         Nome da coluna com as categorias para serem fatiadas no gráfico de pizza.
#     split_col : str, opcional
#         Nome da coluna que separará cada gráfico de pizza.
#     """

#     # Se o dicionário não for passado, gera automaticamente cores fixas
#     if color_dict is None:
#         unique_categories = df[category_col].unique()
#         palette = sns.color_palette('tab10', n_colors=len(unique_categories))
#         color_dict = dict(zip(sorted(unique_categories), palette))

#     # Se split_col for None, gerar apenas um gráfico de pizza para a coluna category_col
#     if split_col is None:
#         counts = df[category_col].value_counts()

#         # Criar o gráfico de pizza
#         plt.figure(figsize=(5, 5))
#         plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90, colors=plt.cm.Set2.colors)
#         plt.title(f'Gráfico de pizza para {category_col}', size=14)
#         plt.tight_layout()
#         plt.show()

#     else:
#         # Obter os valores únicos da coluna split_col
#         split_values = df[split_col].unique()

#         # Configurar o tamanho da figura
#         fig, axes = plt.subplots(nrows=1, ncols=len(split_values), figsize=(5 * len(split_values), 5))

#         # Caso haja apenas um gráfico, convertemos o eixo para uma lista
#         if len(split_values) == 1:
#             axes = [axes]

#         # Gerar um gráfico de pizza para cada valor em split_col
#         for i, value in enumerate(split_values):
#             ax = axes[i]

#             # Filtrar o DataFrame para o valor atual de split_col
#             df_filtered = df[df[split_col] == value]

#             # Contar as frequências de cada categoria
#             counts = df_filtered[category_col].value_counts()

#             # Criar o gráfico de pizza
#             ax.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90, colors=plt.cm.Set2.colors)
#             ax.set_title(f'{split_col}: {value}', size=14)

#         plt.tight_layout()
#         plt.show()
import matplotlib.pyplot as plt
import seaborn as sns

def plot_pie_charts(df, category_col, split_col=None, color_dict=None):
    """
    Plota gráficos de pizza com cores fixas associadas a categorias (clusters).

    Parâmetros:
    -----------
    df : pandas.DataFrame
        O DataFrame contendo os dados.
    category_col : str
        Nome da coluna com as categorias (clusters) para serem exibidas.
    split_col : str, opcional
        Nome da coluna para separar gráficos em múltiplos gráficos.
    color_dict : dict
        Dicionário com as cores fixas para cada categoria.
    """
    if color_dict is None:
        # Se não fornecido, gera automaticamente
        unique_categories = df[category_col].unique()
        palette = sns.color_palette('tab10', len(unique_categories))
        color_dict = dict(zip(sorted(unique_categories), palette))

    if split_col is None:
        counts = df[category_col].value_counts()

        # Corrigido: Assegurar a cor correta usando diretamente os labels do índice
        colors = [color_dict[label] for label in counts.index]

        plt.figure(figsize=(5, 5))
        plt.pie(counts, labels=counts.index, autopct='%1.1f%%',
                startangle=90, colors=colors)
        plt.title(f'Gráfico de pizza para {category_col}', size=14)
        plt.tight_layout()
        plt.show()

    else:
        split_values = df[split_col].unique()
        fig, axes = plt.subplots(nrows=1, ncols=len(split_values),
                                 figsize=(5 * len(split_values), 5))

        if len(split_values) == 1:
            axes = [axes]

        for i, value in enumerate(split_values):
            ax = axes[i]
            df_filtered = df[df[split_col] == value]
            counts = df_filtered[category_col].value_counts()

            # Corrigido: Assegurar cores corretas diretamente do índice
            colors = [color_dict[label] for label in counts.index]

            ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
                   startangle=90, colors=colors)
            ax.set_title(f'{split_col}: {value}', size=14)

        plt.tight_layout()
        plt.show()

import matplotlib.pyplot as plt
import pandas as pd

def plot_stacked_bar_chart(df, category_col, split_col=None, stacked = True):
    """
    Plota gráficos de barras empilhadas. Se `split_col` for fornecido, cria um gráfico
    de barras empilhadas para cada valor único em `split_col`, caso contrário, cria um único gráfico.

    Parâmetros:
    -----------
    df : pandas.DataFrame
        O DataFrame contendo os dados.
    category_col : str
        Nome da coluna com as categorias para serem representadas no gráfico.
    split_col : str, opcional
        Nome da coluna que separará cada gráfico de barras empilhadas.
    """
    if split_col is None:
        # Contar as frequências de cada categoria
        counts = df[category_col].value_counts(normalize=True)  # Normalizado para que a barra tenha a mesma altura

        # Criar o gráfico de barras empilhadas
        counts.plot(kind='bar', stacked=stacked, figsize=(7, 5), color=plt.cm.Set2.colors)
        plt.title(f'Gráfico de barras empilhadas para {category_col}', size=14)
        plt.ylabel('Proporção')
        plt.xlabel(category_col)
        plt.tight_layout()
        plt.show()

    else:
        # Pivotar a tabela para calcular as proporções
        df_grouped = df.groupby([split_col, category_col]).size().unstack().fillna(0)

        # Normalizar para que a barra tenha a mesma altura (1)
        df_grouped_norm = df_grouped.div(df_grouped.sum(axis=1), axis=0)

        # Criar o gráfico de barras empilhadas
        df_grouped_norm.plot(kind='bar', stacked=stacked, figsize=(10, 7), color=plt.cm.Set2.colors)

        # Configurar os rótulos e título
        plt.title(f'Gráfico de barras empilhadas para {split_col} e {category_col}', size=14)
        plt.ylabel('Proporção')
        plt.xlabel(split_col)
        plt.tight_layout()
        plt.show()


def plot_kde(df, var):
  plt.figure(figsize=(10, 6))
  sns.kdeplot(data=df, x=var, fill=True,  palette='Set2')
  plt.title(f'Histograma da Variável {var} por Cluster')
  plt.xlabel(f'{var}')
  plt.ylabel('Frequência')
  plt.show()

#Obter Dados

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Leitura do arquivo Excel com cabeçalho nas linhas 7, 8 e 9
arquivo_excel = 'divulgacao_anos_finais_municipios_2023.xlsx'
df = pd.read_excel(arquivo_excel, header=[6, 7, 8])



# 2. Combinar os níveis do cabeçalho em um único índice
df.columns = ['_'.join([str(item).strip() for item in col if str(item).lower() != 'nan']).strip()
              for col in df.columns.values]

# 3. Remover a primeira linha, que contém os nomes originais dos campos (redundante após combinar o cabeçalho)
df = df.iloc[1:].reset_index(drop=True)

# Exibindo os novos nomes das colunas
print("Nomes das colunas:")
print(df.columns.tolist())


# Supondo que 'df' já seja o DataFrame carregado
# Removendo as ocorrências de 'Unnamed' e 'level' dos nomes das colunas
df.columns = (df.columns
              .str.replace('Unnamed', '', regex=True)
              .str.replace(r'\d+_level_1', '', regex=True)
              .str.replace(r'\d+_level_2', '', regex=True)
              .str.replace('_:', '', regex=True)
              .str.replace('__', '_', regex=True)
              .str.rstrip()  # Remove espaços em branco no final
              .str.replace(r'\s+', '_', regex=True)  # Substitui espaços (um ou mais) por "_"
              .str.replace('-', '_', regex=False)    # Substitui hífens por "_"
              .str.replace('__', '_', regex=True)
              .str.replace('__', '_', regex=True)
              .str.strip('_')
              )


# Exibindo os novos nomes das colunas
print("Novos nomes das colunas:")
print(df.columns.tolist())


# # Verificando o resultado após a limpeza
# print("Visualizando as primeiras linhas após limpeza:")
print(df.head())


In [ ]:
# Verificando o resultado após a limpeza
print("Visualizando as primeiras linhas após limpeza:")
print(df.head())

In [ ]:
df.columns.to_list()

In [ ]:
colunasSelecionadas = [
'Sigla_da_UF',
 'Código_do_Município',
 'Nome_do_Município',
 'Rede',
 'Taxa_de_Aprovação_2005_6º_a_9º_ano',
 'Taxa_de_Aprovação_2023_6º_a_9º_ano',
 'Nota_SAEB_2005_Matemática',
 'Nota_SAEB_2005_Língua_Portuguesa',
 'Nota_SAEB_2023_Matemática',
 'Nota_SAEB_2023_Língua_Portuguesa',
 'IDEB_2005_(N_x_P)',
 'IDEB_2023_(N_x_P)'
]

In [ ]:
# Verificando os valores restantes na coluna 'Rede'
print(df['Rede'].value_counts())

# Filtrando para remover as linhas onde a coluna 'Rede' é 'Pública'
df = df[df['Rede'] != 'Pública']

# Verificando os valores restantes na coluna 'Rede'
print(df['Rede'].value_counts())

In [ ]:
# Extraindo os dados numéricos e removendo linhas com valores ausentes
dados = df[colunasSelecionadas].dropna()
print(dados.describe(include="all"))


In [ ]:
dados.head()

In [ ]:
dados.Sigla_da_UF.value_counts()

In [ ]:
display(df.shape)
display(dados.shape)

In [ ]:
dados.Código_do_Município.nunique()

In [ ]:
dados.columns

In [ ]:
dados.describe(include='all')

In [ ]:
dados.dtypes

In [ ]:
# Convertendo colunas para float, exceto as especificadas
colunas_nao_numericas = ['Sigla_da_UF', 'Código_do_Município', 'Nome_do_Município', 'Rede']
colunas_numericas = [col for col in dados.columns if col not in colunas_nao_numericas]

colunas_numericas


In [ ]:
#dados.dtypes

In [ ]:
for col in colunas_numericas:
    dados[col] = pd.to_numeric(dados[col], errors='coerce')
dados.dtypes

In [ ]:
dados.describe(include='all')

In [ ]:
dados = dados.dropna().reset_index(drop=True)
dados.describe()

In [ ]:
dados.Sigla_da_UF.value_counts()

In [ ]:
dados["Var_IDEB"] = dados['IDEB_2023_(N_x_P)'] / dados['IDEB_2005_(N_x_P)']
dados.describe()

In [ ]:
dados.Sigla_da_UF.value_counts()

In [ ]:
def add_regiao_geografica(df):
  """
  Adiciona uma coluna 'Região Geográfica' ao DataFrame com base na sigla da UF.
  """

  # Dicionário de mapeamento de siglas para regiões
  regioes = {
      'AC': 'Norte', 'AL': 'Nordeste', 'AM': 'Norte', 'AP': 'Norte', 'BA': 'Nordeste',
      'CE': 'Nordeste', 'DF': 'Centro-Oeste', 'ES': 'Sudeste', 'GO': 'Centro-Oeste',
      'MA': 'Nordeste', 'MG': 'Sudeste', 'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
      'PA': 'Norte', 'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'PR': 'Sul',
      'RJ': 'Sudeste', 'RN': 'Nordeste', 'RO': 'Norte', 'RR': 'Norte', 'RS': 'Sul',
      'SC': 'Sul', 'SE': 'Nordeste', 'SP': 'Sudeste', 'TO': 'Norte'
  }

  # Criar a nova coluna 'Região Geográfica' e preencher com base na sigla da UF
  df['Região_Geográfica'] = df['Sigla_da_UF'].map(regioes)

  return df

dados = add_regiao_geografica(dados)



In [ ]:
print(dados.Região_Geográfica.value_counts())

#Preparação de Dados

In [ ]:
columns = ['Var_IDEB']
plot_boxplots(dados, columns, outline=False)

In [ ]:
columns = ['IDEB_2023_(N_x_P)']
plot_boxplots(dados, columns, outline=True)

In [ ]:
columns = ['IDEB_2005_(N_x_P)']
plot_boxplots(dados, columns, outline=True)

In [ ]:
dados.columns

In [ ]:
from sklearn.preprocessing import StandardScaler
import pickle


colunasSelecionadasCluster = [
      'Var_IDEB',
      'IDEB_2005_(N_x_P)',
      'IDEB_2023_(N_x_P)'
]

#Atribuir 0 onde estiver NaN
dados.fillna(0, inplace=True)

scaler = StandardScaler()
dadosPadronizada = scaler.fit_transform(dados[colunasSelecionadasCluster])

#Salvar o scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

#Atualizar o dataframe com as variaveis padronizadas, alterando os seus nomes com algum sufixo
dadosPadronizada = pd.DataFrame(dadosPadronizada, columns=[coluna + '_PADRONIZADA' for coluna in colunasSelecionadasCluster])
dadosPadronizada.head()


In [ ]:
#Criar a lista com os nomes das variaveis padronizadas
colunasSelecionadasPadronizadas = [coluna + '_PADRONIZADA' for coluna in colunasSelecionadasCluster]
colunasSelecionadasPadronizadas

In [ ]:
#concatenar os dois dataframes (normal e padronizada)
dadosPadronizada = pd.concat([dados, dadosPadronizada], axis=1)
dadosPadronizada.columns

In [ ]:
dadosPadronizada[[ 'Var_IDEB', 'Var_IDEB_PADRONIZADA']].describe()

In [ ]:
colunasSelecionadasPadronizadas

#Clusterização

In [ ]:
dadosPadronizada[colunasSelecionadasPadronizadas].describe()

In [ ]:
from tqdm import tqdm
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import pandas as pd

# Supondo que df_cluster é o seu DataFrame
df_cluster = dadosPadronizada.copy()

col_corte = colunasSelecionadasPadronizadas

# Listas para armazenar as somas dos quadrados dentro dos clusters (WCSS) e o coeficiente de silhouette
wcss = []
silhouette_scores = []

maxCluster = 20
# Testando k de 1 a maxCluster
for i in tqdm(range(1, maxCluster)):
    kmeans = KMeans(n_clusters=i, random_state=42, init='k-means++', n_init=10,
                    max_iter=1000, tol=1e-8)
    kmeans.fit(df_cluster[col_corte])

    # Adicionando a inércia (WCSS)
    wcss.append(kmeans.inertia_)



In [ ]:
# Plotando o gráfico do método do cotovelo
plt.figure(figsize=(10, 6))
plt.plot(range(1, maxCluster), wcss, marker='o')
plt.title('Método do Cotovelo para Escolha do Número Ideal de Clusters')
plt.xlabel('Número de Clusters')
plt.ylabel('WCSS')
plt.ylim(ymin=0)
plt.grid(True)
plt.show()


In [ ]:
dadosPadronizada.columns

In [ ]:
df_cluster = dadosPadronizada[colunasSelecionadasCluster+colunasSelecionadasPadronizadas].copy()



i = 4
kmeans = KMeans(n_clusters = i, random_state = 42, init = 'k-means++', n_init = 10,
                max_iter=10000, tol = 1e-8)
kmeans.fit(df_cluster[col_corte])

df_cluster["cluster"] = kmeans.predict(df_cluster[col_corte])
df_cluster["cluster"] = [chr(x+65) for x in df_cluster.cluster]



df_cluster_info = df_cluster.groupby("cluster").median().reset_index()
df_cluster_count = df_cluster.groupby("cluster").count()[col_corte[0]].reset_index()
df_cluster_count.columns = ["cluster", "count"]

df_cluster_table = pd.merge(df_cluster_info, df_cluster_count, how="left", on="cluster")
df_cluster_table

#concatene apenas a coluna "cluster" do df_cluster ao dfPrefeitoBemPadronizada
dadosPadronizadaCluster = pd.concat([dadosPadronizada, df_cluster["cluster"]], axis=1)


In [ ]:
df_cluster_info

In [ ]:
#retirar as colunas colunasSelecionadasPadronizadas do df_cluster_info
df_cluster_info_view = df_cluster_table.drop(colunasSelecionadasPadronizadas, axis=1)

make_spider(df_cluster_info_view)

In [ ]:
df_cluster.columns

In [ ]:
# prompt: criar uma nova coluna DS_cluster, onde irei atribuir um rótulo, baseado no cluster

# Criar um dicionário para mapear os clusters para rótulos
cluster_labels = {
    'A': 'Baixa Performance',
    'B': 'Top 2023',
    'C': 'Melhores Evoluções',
    'D': 'Alta Performance Constante',
    'E': 'E',
    'F': 'F',
    'G': 'G',
    'H': 'H',
    'I': 'I'
     # Adicione mais rótulos conforme necessário
}

# Criar uma nova coluna 'DS_cluster' com os rótulos
dadosPadronizadaCluster['DS_cluster'] = dadosPadronizadaCluster['cluster'].map(cluster_labels)

# Exibir o dataframe com a nova coluna
dadosPadronizadaCluster[['cluster', 'DS_cluster']].head()


In [ ]:
color_dict = {
    'Baixa Performance': 'red',
    'Top 2023': 'lightblue',
    'Melhores Evoluções': 'green',
    'Alta Performance Constante': 'blue'
}

In [ ]:

import plotly.express as px

#colunasSelecionadas

# Supondo que 'col_corte' tenha três colunas
fig = px.scatter_3d(dadosPadronizadaCluster,
                    x=colunasSelecionadasCluster[0],
                    y=colunasSelecionadasCluster[1],
                    z=colunasSelecionadasCluster[2],
                    color='DS_cluster')

# Salvando o gráfico em um arquivo HTML
fig.write_html("grafico_clusters_3d.html")

# Exibindo o gráfico
fig.show()

In [ ]:

import plotly.express as px

#colunasSelecionadas

# Supondo que 'col_corte' tenha três colunas
fig = px.scatter_3d(dadosPadronizadaCluster,
                    x=colunasSelecionadasCluster[0],
                    y=colunasSelecionadasCluster[1],
                    z=colunasSelecionadasCluster[2],
                    color='DS_cluster')

# Salvando o gráfico em um arquivo HTML
fig.write_html("grafico_clusters_3d.html")

# Exibindo o gráfico
fig.show()

In [ ]:
import seaborn as sns
df_plot = dadosPadronizadaCluster.groupby("DS_cluster")[colunasSelecionadasCluster].mean().reset_index()

df_plot_melt = pd.melt(df_plot, id_vars=['DS_cluster'], value_vars= colunasSelecionadasCluster)

sns.barplot(x='DS_cluster', y='value', hue='variable', data=df_plot_melt)
plt.show()

In [ ]:
colunasSelecionadasCluster

In [ ]:
plt.figure(figsize=(10, 6))
Var = 'Var_IDEB'
sns.kdeplot(data=dadosPadronizadaCluster, x=Var, hue='DS_cluster', fill=True,  palette='Set2')
plt.title(f'Histograma da Variável {Var} por Cluster')
plt.xlabel(f'{Var}')
plt.ylabel('Frequência')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
Var = 'IDEB_2023_(N_x_P)'
sns.kdeplot(data=dadosPadronizadaCluster, x=Var, hue='DS_cluster', fill=True,  palette='Set2')
plt.title(f'Histograma da Variável {Var} por Cluster')
plt.xlabel(f'{Var}')
plt.ylabel('Frequência')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
Var = 'IDEB_2005_(N_x_P)'
sns.kdeplot(data=dadosPadronizadaCluster, x=Var, hue='DS_cluster', fill=True,  palette='Set2')
plt.title(f'Histograma da Variável {Var} por Cluster')
plt.xlabel(f'{Var}')
plt.ylabel('Frequência')
plt.show()


In [ ]:
df = dadosPadronizadaCluster.copy()
category_col = 'Rede'
split_col = 'DS_cluster'
plot_pie_charts(df, category_col, split_col)

In [ ]:
df = dadosPadronizadaCluster.copy()
category_col = 'DS_cluster'
split_col = 'Rede'
plot_pie_charts(df, category_col, split_col, color_dict)

In [ ]:
dadosPadronizadaCluster.columns

In [ ]:
df = dadosPadronizadaCluster.copy()
category_col = 'Região_Geográfica'
split_col = 'DS_cluster'
plot_pie_charts(df, category_col, split_col)

In [ ]:
df = dadosPadronizadaCluster.copy()
category_col = 'DS_cluster'
split_col = 'Região_Geográfica'
plot_pie_charts(df, category_col, split_col)

In [ ]:
df = dadosPadronizadaCluster.copy()
category_col = 'Sigla_da_UF'
split_col = 'DS_cluster'
plot_pie_charts(df, category_col, split_col)

In [ ]:
df = dadosPadronizadaCluster.copy()
category_col = 'DS_cluster'
split_col = 'Sigla_da_UF'
plot_pie_charts(df, category_col, split_col)

# Regras de Associação

In [ ]:
dadosPadronizadaCluster.head()


In [ ]:
# Função para criar faixas de quartis
def criar_faixas_quartis(df, colunas):
    for coluna in colunas:
        if pd.api.types.is_numeric_dtype(df[coluna]):
            quartis = df[coluna].quantile([0.25, 0.5, 0.75])
            df[coluna + '_Quartis'] = pd.cut(df[coluna], bins=[-float('inf'), quartis[0.25], quartis[0.5], quartis[0.75], float('inf')],
                                              labels=['Q1', 'Q2', 'Q3', 'Q4'], include_lowest=True, duplicates='drop')
    return df

# Colunas numéricas para criar faixas de quartis
colunas_numericas = ['Taxa_de_Aprovação_2005_6º_a_9º_ano', 'Taxa_de_Aprovação_2023_6º_a_9º_ano',
                    'Nota_SAEB_2005_Matemática', 'Nota_SAEB_2005_Língua_Portuguesa',
                    'Nota_SAEB_2023_Matemática', 'Nota_SAEB_2023_Língua_Portuguesa',
                    'IDEB_2005_(N_x_P)', 'IDEB_2023_(N_x_P)', 'Var_IDEB']


# Aplicar a função para criar as faixas de quartis
dadosPadronizadaCluster = criar_faixas_quartis(dadosPadronizadaCluster, colunas_numericas)

# Exibir as primeiras linhas do DataFrame com as novas colunas
print(dadosPadronizadaCluster.head())


In [ ]:
dadosPadronizadaCluster.columns.to_list()

In [ ]:
# Renomear as colunas
novos_nomes = {
    'Taxa_de_Aprovação_2005_6º_a_9º_ano': 'TxApr_2005',
    'Taxa_de_Aprovação_2023_6º_a_9º_ano': 'TxApr_2023',
    'Nota_SAEB_2005_Matemática': 'Nota_Mat_2005',
    'Nota_SAEB_2005_Língua_Portuguesa': 'Nota_Por_2005',
    'Nota_SAEB_2023_Matemática': 'Nota_Mat_2023',
    'Nota_SAEB_2023_Língua_Portuguesa': 'Nota_Por_2023',
    'IDEB_2005_(N_x_P)': 'IDEB_2005',
    'IDEB_2023_(N_x_P)': 'IDEB_2023',
    'Var_IDEB': 'Var_IDEB',
    'Região_Geográfica': 'Regiao',
    'Var_IDEB_PADRONIZADA': 'Var_IDEB_Pad',
    'IDEB_2005_(N_x_P)_PADRONIZADA': 'IDEB_2005_Pad',
    'IDEB_2023_(N_x_P)_PADRONIZADA': 'IDEB_2023_Pad',
    'Taxa_de_Aprovação_2005_6º_a_9º_ano_Quartis': 'TxApr_2005_Qrt',
    'Taxa_de_Aprovação_2023_6º_a_9º_ano_Quartis': 'TxApr_2023_Qrt',
    'Nota_SAEB_2005_Matemática_Quartis': 'Nota_Mat_2005_Qrt',
    'Nota_SAEB_2005_Língua_Portuguesa_Quartis': 'Nota_Por_2005_Qrt',
    'Nota_SAEB_2023_Matemática_Quartis': 'Nota_Mat_2023_Qrt',
    'Nota_SAEB_2023_Língua_Portuguesa_Quartis': 'Nota_Por_2023_Qrt',
    'IDEB_2005_(N_x_P)_Quartis': 'IDEB_2005_Qrt',
    'IDEB_2023_(N_x_P)_Quartis': 'IDEB_2023_Qrt',
    'Var_IDEB_Quartis': 'Var_IDEB_Qrt'
}

dadosPadronizadaCluster = dadosPadronizadaCluster.rename(columns=novos_nomes)
dadosPadronizadaCluster.columns


In [ ]:
dadosPadronizadaCluster.head()

In [ ]:
dadosPadronizadaCluster.columns.to_list()

In [ ]:

# Lista de colunas que você deseja manter
colunas_desejadas = [
    'Sigla_da_UF',
    'Código_do_Município',
    'Nome_do_Município',
    'Rede',
    'Regiao',
    'DS_cluster',
    'TxApr_2005_Qrt',
    'Nota_Mat_2005_Qrt',
    'Nota_Por_2005_Qrt',
    'IDEB_2005_Qrt'
]

# Selecionando apenas as colunas desejadas
dadosPadronizadaCluster_filtrado = dadosPadronizadaCluster[colunas_desejadas]

# Exibindo as primeiras linhas do DataFrame filtrado para confirmar a operação
print(dadosPadronizadaCluster_filtrado.head())


In [ ]:
dadosPadronizadaCluster_filtrado.columns.to_list()

In [ ]:
# Lista de colunas que não serão transformadas
colunas_excluir = ['Código_do_Município', 'Nome_do_Município']

# Criar um novo DataFrame que exclui as colunas especificadas
df_subconjunto = dadosPadronizadaCluster_filtrado.drop(columns=colunas_excluir)

# Aplicar one-hot encoding a todas as colunas restantes
# pd.get_dummies() irá identificar automaticamente as colunas categóricas e
# criar as variáveis dummy correspondentes.
df_onehot = pd.get_dummies(df_subconjunto)

# Exibir as primeiras linhas do DataFrame resultante para conferência
print(df_onehot.head())

In [ ]:
!pip install pyvis
!pip install mlxtend
!pip install pandas
!pip install matplotlib
!pip install networkx
!pip install numpy


In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import pandas as pd
import random
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns

In [ ]:
df_onehot.columns.to_list()

In [ ]:
from mlxtend.frequent_patterns import association_rules, apriori

frequent_items = apriori(df_onehot, min_support = 0.05,use_colnames = True)
frequent_items

In [ ]:
rules = association_rules(frequent_items, metric = "lift", min_threshold = 1)
rules.sort_values('lift', ascending = False, inplace = True)
rules

In [ ]:
grules = rules[rules['lift'] > 1]
grules.shape

In [ ]:

rules_consequent_1 = grules[grules['consequents'].apply(lambda x: len(x) == 1)]
display(rules_consequent_1)


In [ ]:

rules_antecedent_1 = grules[grules['antecedents'].apply(lambda x: len(x) == 1)]
display(rules_antecedent_1)

In [ ]:
rules_consequent_1_DS_cluster = rules[
    rules['antecedents'].apply(lambda x: len(x) == 1 and any('DS_cluster_B' in str(item) for item in x))
]

rules_consequent_1_DS_cluster = rules_consequent_1_DS_cluster[
    rules['confidence'].apply(lambda x: x>= 0.3)
]

rules_filtered = rules_consequent_1_DS_cluster[
    rules_consequent_1_DS_cluster['consequents'].apply(lambda x: len(x) <= 2)
]

display(rules_filtered)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

pd.set_option('display.max_colwidth', None)

import pandas as pd
import networkx as nx
from pyvis.network import Network
from google.colab import files

def make_graph_2(grules):
    G = nx.DiGraph()

    # Adicionando nós e arestas
    for index, row in grules.iterrows():
        # Criando rótulos para os nós de antecedente e consequente
        antecedent_items = sorted(map(str, row['antecedents']))
        consequent_items = sorted(map(str, row['consequents']))
        antecedent_label = ', '.join(antecedent_items)
        consequent_label = ', '.join(consequent_items)

        # Adicionando os nós de antecedente e consequente se ainda não existirem
        if not G.has_node(antecedent_label):
            G.add_node(antecedent_label, title=antecedent_label)
        if not G.has_node(consequent_label):
            G.add_node(consequent_label, title=consequent_label)

        # Adicionando uma aresta entre o antecedente e o consequente
        G.add_edge(
            antecedent_label,
            consequent_label,
            title=f"Confiança: {row['confidence']:.2f}, Lift: {row['lift']:.2f}",
            confidence=row['confidence'],
            lift=row['lift']
        )

    # Criando a rede com pyvis
    net = Network(
        height="750px",
        width="100%",
        bgcolor="#222222",
        font_color="white",
        notebook=True,
        directed=True,
        cdn_resources='in_line'
    )

    # Convertendo o grafo networkx para pyvis
    net.from_nx(G)

    # Ajustando estilos e interações
    for node in net.nodes:
        node_label = node['label']
        node['title'] = node_label
        node['shape'] = 'box'  # Opcional: para melhor visualização de conjuntos de itens

        # Verificando se o nó representa um único item
        if ',' not in node_label:
            # Verificando se o item começa com 'DS_cluster'
            if node_label.startswith('DS_cluster'):
                # Destaque para os nós que atendem ao critério
                node['color'] = 'red'         # Cor diferenciada
                node['font'] = {'size': 14}   # Fonte maior
                node['borderWidth'] = 2       # Borda mais espessa
                node['size'] = 40             # Nó maior
            else:
                # Estilo padrão para nós individuais
                node['color'] = '#1f78b4'
                node['size'] = 5
                node['font'] = {'size': 2}   # Fonte menor
        else:
            # Estilo para nós que representam múltiplos itens
            node['color'] = '#6a3d9a'
            node['size'] = 5
            node['font'] = {'size': 2}   # Fonte menor

    for edge in net.edges:
        edge['title'] = edge['title']
        edge['value'] = edge['confidence']  # Ajusta a espessura da aresta com base na confiança

    # Habilitando a física para movimentação dinâmica dos nós
    net.show_buttons(filter_=['physics'])
    net.toggle_physics(True)

    # Salvando e fazendo o download do grafo
    net.save_graph("supermarket_associations_directed.html")
    files.download("supermarket_associations_directed.html")


In [ ]:
make_graph2(rules_filtered)